<a href="https://colab.research.google.com/github/nandakumar3261/ai-mentor-portfolio-Nandakumar/blob/main/Day2_AN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai pydantic

In [2]:
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [3]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [4]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            # Retry once with the broken JSON in the prompt
            fix_prompt = (f'Fix this JSON to match schema. Errors: {e}. '
                          f'Original: {resp.text}')
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)

In [9]:
with open('/content/sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]

print(f'Loaded {len(resumes)} sample résumés')

results = []
for i, r in enumerate(resumes[:3]):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'\nRésumé {i+1}: {parsed.name} — {len(parsed.skills)} skills, '
              f'{parsed.experience_years} years exp')
    except Exception as e:
        print(f'\nRésumé {i+1}: FAILED — {type(e).__name__}: {str(e)[:200]}')

# Print full Second result
if results:
    print('\n=== Full first result ===')
    print(results[1].model_dump_json(indent=2))

Loaded 5 sample résumés

Résumé 1: Ravi Kumar — 6 skills, 0.25 years exp

Résumé 2: Sneha Reddy — 6 skills, 0.115 years exp

Résumé 3: Arun Pillai — 8 skills, 0.5 years exp

=== Full first result ===
{
  "name": "Sneha Reddy",
  "email": "sneha.r@example.com",
  "phone": "+91-9123456789",
  "education": [
    {
      "degree": "B.Tech Electronics & Communication",
      "institution": "Aditya University",
      "year": 2026
    },
    {
      "degree": "Intermediate",
      "institution": "Sri Chaitanya",
      "year": 2022
    }
  ],
  "skills": [
    "C++",
    "Python",
    "MATLAB",
    "Embedded C",
    "VLSI Design",
    "Verilog"
  ],
  "projects": [
    "IoT-based water quality monitoring system (Arduino + ESP32)",
    "VLSI implementation of a 4-bit ALU using Verilog",
    "Optimised PCB layout for industrial sensor module"
  ],
  "experience_years": 0.115
}
